# Recursive Directory Traversal with Event Handlers

Directory trees are inherently recursive: a directory may contain files and other directories, which in turn may contain more files and directories.

This recursive definition makes directory traversal a ideal candidate for recursive processing in the real world since users interact with folders and files on a daily basis.

Note that other items, e.g. symlinks, are omitted here for simplicity but can be added trivially.

To process a directorystructure, our base case is when the current path is a file or a directory with no entries.

The recursive case occurs when a directory contains one or more entries, each of which must be handled recursively.

Here is an example simple directory tree for reference:

```
root/
+-- file1.txt
+-- dirA/
|   +-- file2.txt
|   +-- dirB/
|       +-- file3.txt
+-- dirC/
    +-- file4.txt
```
In Unix, you can create the above structure using `mkdir`. We'll show this below.


We want to process all paths in this tree, visiting each directory and file in an orderly and extensible fashion.

\subsection{Core Recursive Implementation (DFS)}

The first version of our traversal uses a depth-first search (DFS) approach, which is a natural fit for recursion.
We process a directory’s entries before returning to its parent.

## Install prerequisites / create test data

In [1]:
!mkdir -p root/dirA/dirB root/dirC
!touch root/file1.txt root/dirA/file2.txt root/dirA/dirB/file3.txt root/dirC/file4.txt

In [2]:
!apt install tree


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (326 kB/s)
Selecting previously unselected package tree.
(Reading database ... 125082 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
!tree root

root
├── dirA
│   ├── dirB
│   │   └── file3.txt
│   └── file2.txt
├── dirC
│   └── file4.txt
└── file1.txt

3 directories, 4 files


## Problem Statement

In this example, we explore how to recursively walk a directory tree while allowing custom processing of traversal events.
The goal is to create a reusable framework that visits each file and directory under a given root, using a user-defined handler to respond to events like entering a directory, encountering a file, or dealing with an error.

Python’s built-in \verb|os.walk()| provides an interface that works for most users.
Contrasted with this version, which returns tuples in a procedural style for convenience sake, our version separates traversal logic from processing logic.
This enables better code reuse, modularity, and extensibility.
We also show how this recursive structure maps naturally onto a tree traversal and how object-oriented programming (OOP) helps manage side effects in a clean way.


## Recursive Structure of the Problem

Directory trees are inherently recursive: a directory may contain files and other directories, which in turn may contain more files and directories. This recursive definition makes directory traversal a perfect candidate for recursive processing.

% Note that other items, e.g. symlinks, are omitted here for simplicity but can be added trivially.

To process such a structure, our base case is when the current path is a file or a directory with no entries. The recursive case occurs when a directory contains one or more entries, each of which must be handled recursively.

Here is a simple directory tree for reference:

```
root/
+-- file1.txt
+-- dirA/
|   +-- file2.txt
|   +-- dirB/
|       +-- file3.txt
+-- dirC/
```

% Add something about the tree command.

We want to process all paths in this tree, visiting each directory and file in an orderly and extensible fashion.

## Core Recursive Implementation (DFS)

The first version of our traversal uses a depth-first search (DFS) approach, which is a natural fit for recursion. We process a directory’s entries before returning to its parent.

In [4]:
import os
from typing import Protocol, Union

class FileSystemEventWalker(Protocol):
    def on_enter_directory(self, path: str): ...
    def on_exit_directory(self, path: str): ...
    def on_file(self, path: str): ...
    def on_error(self, path: str, error: Exception): ...

def walk_dfs(path: str, handler: FileSystemEventWalker):
    try:
        if os.path.isdir(path):
            handler.on_enter_directory(path)
            try:
                entries = os.listdir(path)
            except Exception as e:
                handler.on_error(path, e)
                return
            for entry in entries:
                full_path = os.path.join(path, entry)
                walk_dfs(full_path, handler)
            handler.on_exit_directory(path)
        elif os.path.isfile(path):
            handler.on_file(path)
        else:
            handler.on_error(path, Exception(f"Unknown path/exists issue: {path}"))
    except Exception as e:
        handler.on_error(path, e)

This code recursively traverses a directory tree.
If the current path is a directory, it signals the provided handler via `on_enter_directory()`, recursively walks its contents, and finally calls `on_exit_directory()`.
If the path is a file, it triggers `on_file()`.
Any errors -- or paths that cannot be processed -- are caught and delegated to `on_error()`.

The `Protocol` base class is used here to define the expected interface for the event handler.


## Basic Handler

To separate traversal from any potential side effects, we introduce a handler interface.
The user provides an object with methods to be called at key moments during the walk.
This pattern resembles SAX-style \emph{stream parsing} -- used in processing XML documents -- where traversal and processing are decoupled.
Stream parsing has a notable advantage: Until the handler actually does anything with the streaming events, no storage is expended.

In addition, even though the *thinking* is recursive, the handler has the responsibility for managing whatever state/context it wishes to maintain (e.g. what position in the directory hierarchy, etc.)

In [5]:
class BasicHandler(Protocol):
    def __init__(self, indent=2):
        self.level = 0
        self.indent = indent

    def tab(self):
        indent_text = " " * self.indent * self.level
        print(indent_text, end='')

    def on_enter_directory(self, path: str):
        self.level += 1
        self.tab()
        print(f"Entering: {path}")

    def on_exit_directory(self, path: str):
        self.tab()
        self.level -= 1
        print(f"Exiting: {path}")

    def on_file(self, path: str):
        self.level += 1
        self.tab()
        self.level -= 1
        print(f"File: {path}")

    def on_error(self, path: str, error: Exception):
        print(f"Error at {path}: {error}")

This default handler prints a message whenever a directory or file is visited.
It can be subclassed or replaced by custom logic without changing the recursive walk behavior itself.
This makes the solution modular and flexible.


## A Quick Check to Play Events with Test Folder

In [6]:
def play_events(directory):
    handler = BasicHandler()
    walk_dfs(directory, handler)


In [7]:
play_events("root")

  Entering: root
    File: root/file1.txt
    Entering: root/dirA
      Entering: root/dirA/dirB
        File: root/dirA/dirB/file3.txt
      Exiting: root/dirA/dirB
      File: root/dirA/file2.txt
    Exiting: root/dirA
    Entering: root/dirC
      File: root/dirC/file4.txt
    Exiting: root/dirC
  Exiting: root


In [8]:
play_events("root/file1.txt")

  File: root/file1.txt


In [9]:
play_events("root/file2.txt") # Should give us an Exception object...

Error at root/file2.txt: Unknown path/exists issue: root/file2.txt


## Writing Handler to Count Files

One advantage of this design is the ability to customize behavior by writing your own handler class.
For example, the following handler counts all files encountered during the traversal.


In [10]:
class FileCounterHandler:
    def __init__(self, max_depth=0):
        self.file_count = 0
        self.folder_count = 0
        self.max_depth = max_depth
        self.depth = 0

    # This method is NOT part of the handler protocol
    # it is an internal method so we can minimize repetition of the actual check

    def directory_depth_reached(self):
        return self.max_depth > 0 and self.depth >= self.max_depth

    def on_enter_directory(self, path: str):
        if not self.directory_depth_reached():
            self.folder_count += 1
            self.depth += 1

    def on_exit_directory(self, path: str):
        if not self.directory_depth_reached():
            self.depth -= 1

    def on_file(self, path: str):
        if not self.directory_depth_reached():
            self.file_count += 1

    def on_error(self, path: str, error: Exception):
        print(f"Error: {path} - {error}")

A simple example of how to walk and count files and directories within a hierarchy.

To make things interesting, the user can specify a positive  `max_depth` to indicate the maximum depth to visit folders recursively. A for `max_depth <= 0` it is assumed

In [11]:
from collections import namedtuple

# namedtuple: Python's "record" type (a.k.a. a data class)
Counts = namedtuple('Counts', ['files', 'folders'])

def count_files_and_folders(directory, depth=0):
    handler = FileCounterHandler(depth)
    walk_dfs(directory, handler)
    return Counts(files=handler.file_count, folders=handler.folder_count)

In [17]:
print("Note that depth <= assumes infinite depth")
print()

for depth in range(5, -1, -1):
    root_counts = count_files_and_folders("root", depth)
    print(f"Total number of files/folders at depth {depth}: {root_counts}")



Note that depth=0 assumes infinite depth

Total number of files/folders at depth 5: Counts(files=4, folders=4)
Total number of files/folders at depth 4: Counts(files=4, folders=4)
Total number of files/folders at depth 3: Counts(files=1, folders=3)
Total number of files/folders at depth 2: Counts(files=1, folders=2)
Total number of files/folders at depth 1: Counts(files=0, folders=1)
Total number of files/folders at depth 0: Counts(files=4, folders=4)


## Walking Breadth First or Depth First

In [13]:
from collections import deque

def walk_bfs(root: str, handler: FileSystemEventWalker):
    queue = deque([root])
    while queue:
        path = queue.popleft()
        try:
            if os.path.isdir(path):
                handler.on_enter_directory(path)
                try:
                    entries = os.listdir(path)
                    for entry in entries:
                        queue.append(os.path.join(path, entry))
                except Exception as e:
                    handler.on_error(path, e)
                handler.on_exit_directory(path)
            else:
                handler.on_file(path)
        except Exception as e:
            handler.on_error(path, e)

This code performs an iterative traversal using a Python collections `deque`, which supports doubly-ended queues.

Each time a directory is visited, its entries are added to the queue.

Files are processed as they appear within a folder and is similar to how `os.walk()` operates in Python.

This version is useful when you want to process items in level-order, such as for UI rendering or progressive scanning.

## Generalize the walker to DFS/BFS, user selectable


In [14]:
def walk(path: str, handler: FileSystemEventWalker, strategy: str = "dfs"):
    if strategy == "dfs":
        walk_dfs(path, handler)
    elif strategy == "bfs":
        walk_bfs(path, handler)
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

## Conclusion

This example demonstrates how recursion can be applied to a real-world problem with tree structure.
The key idea is that recursive descent matches the problem domain—directories contain subdirectories and files—allowing us to process each entry with a simple and elegant recursive call.

Equally important is the modularity introduced through object-oriented design.
By isolating the traversal mechanism from the behavior triggered at each node, we gain reusability and extensibility.
Handlers can perform tasks like counting, logging, filtering, or validating, all without touching the recursive logic.

This pattern is broadly applicable to many domains: abstract syntax trees, organizational hierarchies, document outlines, and more.
The recursive directory walker example shows how combining recursion and modular design leads to expressive, maintainable software.
